# Extracting Alpha from Financial Analyst Networks

## Description
This notebook implements the momentum trading strategy based on the coverage network of financial analysts as described in the paper "Extracting Alpha from Financial Analyst Networks" by Dragos Gorduza, Yaxuan Kong, Xiaowen Dong, and Stefan Zohren. The strategy leverages a graph attention network to aggregate information from individual firm features and signals from neighboring firms in a node-level forecasting task.

**Paper Citation:**
Dragos Gorduza, Yaxuan Kong, Xiaowen Dong, Stefan Zohren. "Extracting Alpha from Financial Analyst Networks". arXiv:2410.20597 [q-fin.TR], 27 Oct 2024.

**Abstract:**
We investigate the effectiveness of a momentum trading signal based on the coverage network of financial analysts. This signal builds on the key information-brokerage role financial sell-side analysts play in modern stock markets. The baskets of stocks covered by each analyst can be used to construct a network between firms whose edge weights represent the number of analysts jointly covering both firms. Although the link between financial analysts coverage and co-movement of firms' stock prices has been investigated in the literature, little effort has been made to systematically learn the most effective combination of signals from firms covered jointly by analysts in order to benefit from any spillover effect. To fill this gap, we build a trading strategy which leverages the analyst coverage network using a graph attention network. More specifically, our model learns to aggregate information from individual firm features and signals from neighbouring firms in a node-level forecasting task. We develop a portfolio based on those predictions which we demonstrate to exhibit an annualized returns of 29.44% and a Sharpe ratio of 4.06 substantially outperforming market baselines and existing graph machine learning based frameworks. We further investigate the performance and robustness of this strategy through extensive empirical analysis. Our paper represents one of the first attempts in using graph machine learning to extract actionable knowledge from the analyst coverage network for practical financial applications.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 — Trading Context & Objectives

In [ ]:
# Configuration
UNIVERSE = ['AAPL', 'MSFT']
START_DATE = '2020-01-01'
END_DATE = '2024-10-27'
LOOKBACK_PERIOD = 252
POSITION_SIZE = 0.02

# Hypothesis
# The strategy hypothesizes that stocks covered by the same financial analysts exhibit momentum effects.
# By leveraging a graph attention network, we can aggregate information from individual firm features
# and signals from neighboring firms to predict future returns.


## Phase 2 — Data Download & Feature Computation

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Download data
data = yf.download(UNIVERSE, start=START_DATE, end=END_DATE, group_by='ticker')
prices = pd.concat({ticker: data[ticker]['Adj Close'] for ticker in UNIVERSE}, axis=1)

# Compute returns
returns = prices.pct_change().dropna()

# Cross-sectional normalization
normalized_returns = (returns - returns.mean(axis=1).values.reshape(-1, 1)) / returns.std(axis=1).values.reshape(-1, 1)

## Phase 3 — Signal Generation & Portfolio Construction

In [ ]:
# Signal generation
momentum_signals = normalized_returns.rolling(window=LOOKBACK_PERIOD).mean().dropna()

# Position sizing
positions = momentum_signals.rank(axis=1, pct=True) * POSITION_SIZE

# Portfolio construction
portfolio_weights = positions.iloc[-1]

## Phase 4 — Vectorized Backtest

In [ ]:
# Vectorized backtest
daily_returns = returns.loc[portfolio_weights.index]
portfolio_returns = (daily_returns * portfolio_weights).sum(axis=1)
cumulative_returns = (1 + portfolio_returns).cumprod()

## Phase 5 — Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import norm

# Performance metrics
annual_return = portfolio_returns.mean() * 252
annual_volatility = portfolio_returns.std() * np.sqrt(252)
sharpe_ratio = annual_return / annual_volatility
sortino_ratio = annual_return / portfolio_returns[portfolio_returns < 0].std() * np.sqrt(252)
max_drawdown = (cumulative_returns / cumulative_returns.cummax() - 1).min()
calmar_ratio = annual_return / abs(max_drawdown)

# Print metrics
print(f'Annual Return: {annual_return:.2%}')
print(f'Annual Volatility: {annual_volatility:.2%}')
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')

# Plot equity curve
plt.plot(cumulative_returns.index, cumulative_returns)
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.show()

## Phase 6 — Monitoring Stub

In [ ]:
def monitor_portfolio(live_data):
    # Placeholder for live data monitoring
    daily_pnl = live_data.pct_change().dropna().sum(axis=1).iloc[-1]
    current_positions = positions.iloc[-1]
    print(f'Daily P&L: {daily_pnl:.2%}')
    print('Current Positions:')
    print(current_positions)

# Example usage
monitor_portfolio(prices)